# `to_vtk`

Exports a 2D or 3D NumPy array to a VTK ImageData (`.vti`) file for visualization in ParaView or PyVista.

In [1]:
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv

import porespy as ps

ps.visualization.set_mpl_style()

## `im` and `filename`

Generate a 3D porous image and export it to a `.vti` file. The `.vti` extension is appended automatically.

In [2]:
im = ps.generators.blobs(shape=[40, 40, 40], porosity=0.6, seed=0)

with tempfile.TemporaryDirectory() as tmpdir:
    fname = os.path.join(tmpdir, "blobs_3d")
    ps.io.to_vtk(im=im, filename=fname)
    vti_path = fname + ".vti"
    print(f"File size: {os.path.getsize(vti_path):,} bytes")
    grid = pv.read(vti_path)

print("Dimensions:", grid.dimensions)
print("Spacing:", grid.spacing)
print("Arrays:", list(grid.cell_data.keys()))

File size: 64,433 bytes
Dimensions: (41, 41, 41)
Spacing: (1.0, 1.0, 1.0)
Arrays: ['im']


## `voxel_size`

Setting `voxel_size` records the physical side length of each cubic voxel in the `.vti` metadata:

In [3]:
with tempfile.TemporaryDirectory() as tmpdir:
    fname = os.path.join(tmpdir, "blobs_scaled")
    ps.io.to_vtk(im=im, filename=fname, voxel_size=5)
    grid_scaled = pv.read(fname + ".vti")

print("Spacing:", grid_scaled.spacing)

Spacing: (5.0, 5.0, 5.0)


## `divide`

For large 3D datasets, `divide=True` splits the volume in half along the z-axis into two `.vti` files (`<filename>_1.vti` and `<filename>_2.vti`) with matching origins:

In [4]:
with tempfile.TemporaryDirectory() as tmpdir:
    fname = os.path.join(tmpdir, "blobs_divided")
    ps.io.to_vtk(im=im, filename=fname, divide=True)
    g1 = pv.read(fname + "_1.vti")
    g2 = pv.read(fname + "_2.vti")

print("Part 1 dimensions:", g1.dimensions, "origin:", g1.origin)
print("Part 2 dimensions:", g2.dimensions, "origin:", g2.origin)

Part 1 dimensions: (41, 41, 21) origin: (0.0, 0.0, 0.0)
Part 2 dimensions: (41, 41, 21) origin: (0.0, 0.0, 20.0)


## `downsample`

For very large volumes, `downsample=True` reduces the image by half along each axis and doubles the effective voxel spacing:

In [5]:
with tempfile.TemporaryDirectory() as tmpdir:
    fname = os.path.join(tmpdir, "blobs_downsampled")
    ps.io.to_vtk(im=im, filename=fname, downsample=True, voxel_size=1)
    grid_ds = pv.read(fname + ".vti")

print("Downsampled dimensions:", grid_ds.dimensions)
print("Downsampled spacing:", grid_ds.spacing)

Downsampled dimensions: (21, 21, 21)
Downsampled spacing: (2.0, 2.0, 2.0)


## `vox`

Setting `vox=True` casts numeric arrays to `int8` before writing, which reduces the output file size when full floating-point precision is not needed:

In [6]:
dt = ps.tools.get_edt()(im)

with tempfile.TemporaryDirectory() as tmpdir:
    f_float = os.path.join(tmpdir, "dt_float")
    f_vox = os.path.join(tmpdir, "dt_vox")
    ps.io.to_vtk(im=dt, filename=f_float, vox=False)
    ps.io.to_vtk(im=dt, filename=f_vox, vox=True)
    size_float = os.path.getsize(f_float + ".vti")
    size_vox = os.path.getsize(f_vox + ".vti")

print(f"vox=False file size: {size_float:,} bytes")
print(f"vox=True  file size: {size_vox:,} bytes")

vox=False file size: 256,436 bytes
vox=True  file size: 64,433 bytes
